# Efeito de Esboço a Lápis

Implementação de efeito de esboço a lápis em uma imagem através dos seguintes passos:
1. Converter a imagem colorida para níveis de cinza
2. Aplicar filtro de desfoque gaussiano (máscara 21×21)
3. Dividir a imagem em tons de cinza pela versão desfocada

In [ ]:
import cv2
import numpy as np

## Funções Implementadas Manualmente

In [ ]:
def converter_para_cinza(img_colorida):
    """
    Converte imagem colorida para escala de cinza usando a fórmula dos pesos:
    Gray = 0.299*R + 0.587*G + 0.114*B
    """
    # OpenCV carrega em BGR
    b = img_colorida[:, :, 0].astype(np.float64)
    g = img_colorida[:, :, 1].astype(np.float64)
    r = img_colorida[:, :, 2].astype(np.float64)
    
    # Aplicar pesos e somar
    cinza = 0.299 * r + 0.587 * g + 0.114 * b
    
    # Converter para uint8
    return np.clip(cinza, 0, 255).astype(np.uint8)

In [ ]:
def criar_kernel_gaussiano(tamanho, sigma=0):
    """
    Cria um kernel gaussiano 2D manualmente.
    tamanho: dimensão do kernel (ex: 21 para 21x21)
    sigma: desvio padrão (se 0, calcula automaticamente)
    """
    if sigma == 0:
        sigma = 0.3 * ((tamanho - 1) * 0.5 - 1) + 0.8
    
    # Criar eixos coordenados centrados no meio do kernel
    ax = np.arange(-tamanho // 2 + 1., tamanho // 2 + 1.)
    xx, yy = np.meshgrid(ax, ax)
    
    # Calcular gaussiana 2D
    kernel = np.exp(-(xx**2 + yy**2) / (2. * sigma**2))
    
    # Normalizar para soma = 1
    kernel = kernel / np.sum(kernel)
    
    return kernel

In [ ]:
def aplicar_gaussiana(img, kernel):
    """
    Aplica convolução com kernel gaussiano manualmente.
    Usa padding de borda replicada.
    """
    k_size = kernel.shape[0]
    pad = k_size // 2
    
    # Padding da imagem (replicar bordas)
    img_pad = np.pad(img.astype(np.float64), pad, mode='edge')
    
    h, w = img.shape
    resultado = np.zeros_like(img, dtype=np.float64)
    
    # Convolução manual
    for i in range(h):
        for j in range(w):
            # Extrair região de interesse
            regiao = img_pad[i:i+k_size, j:j+k_size]
            # Aplicar kernel e somar
            resultado[i, j] = np.sum(regiao * kernel)
    
    return np.clip(resultado, 0, 255).astype(np.uint8)

In [ ]:
def divisor_dodge(cinza, desfocada, escala=255.0):
    """
    Aplica o efeito de divisão (color dodge) para realçar contornos.
    Fórmula: resultado = cinza / (255 - desfocada) * 255
    """
    # Converter para float para evitar overflow
    cinza_f = cinza.astype(np.float64)
    desfocada_f = desfocada.astype(np.float64)
    
    # Evitar divisão por zero
    denominador = 255.0 - desfocada_f
    denominador = np.where(denominador == 0, 1e-10, denominador)
    
    # Aplicar fórmula do color dodge
    esboco = (cinza_f / denominador) * escala
    
    return np.clip(esboco, 0, 255).astype(np.uint8)

## Processamento da Imagem

In [ ]:
# Carregar imagem usando OpenCV
imagem = cv2.imread('portrait-handsome-smiling-stylish-young-man-model-wearing-jeans-clothes-sunglasses-fashion-man.jpg')

if imagem is None:
    raise ValueError("Erro ao carregar a imagem!")

print(f"Imagem carregada: {imagem.shape}")

### Passo (i): Converter para níveis de cinza

In [ ]:
cinza = converter_para_cinza(imagem)
print(f"Imagem em tons de cinza: {cinza.shape}")

# Salvar resultado intermediário
cv2.imwrite('resultado_cinza.jpg', cinza)

### Passo (ii): Aplicar filtro gaussiano (21×21)

In [ ]:
# Criar kernel gaussiano 21x21
kernel = criar_kernel_gaussiano(21)
print(f"Kernel gaussiano criado: {kernel.shape}")
print(f"Soma do kernel: {np.sum(kernel):.6f}")

# Aplicar desfoque gaussiano
desfocada = aplicar_gaussiana(cinza, kernel)
print(f"Imagem desfocada: {desfocada.shape}")

# Salvar resultado intermediário
cv2.imwrite('resultado_desfocada.jpg', desfocada)

### Passo (iii): Dividir cinza pela versão desfocada

In [ ]:
# Aplicar efeito de divisão para realçar contornos
esboco = divisor_dodge(cinza, desfocada)
print(f"Esboço final: {esboco.shape}")

# Salvar resultado final
cv2.imwrite('resultado_esboco.jpg', esboco)

print("\nProcessamento concluído!")
print("Arquivos salvos:")
print("  - resultado_cinza.jpg (etapa i)")
print("  - resultado_desfocada.jpg (etapa ii)")
print("  - resultado_esboco.jpg (etapa iii - resultado final)")

## Visualização dos Resultados

In [ ]:
import matplotlib.pyplot as plt

# Carregar resultados para visualização
img_original = cv2.cvtColor(imagem, cv2.COLOR_BGR2RGB)
img_cinza = cv2.imread('resultado_cinza.jpg', 0)
img_desfocada = cv2.imread('resultado_desfocada.jpg', 0)
img_esboco = cv2.imread('resultado_esboco.jpg', 0)

# Criar figura com 4 subplots
plt.figure(figsize=(16, 4))

# Imagem original colorida
plt.subplot(1, 4, 1)
plt.imshow(img_original)
plt.title('Original (Colorida)')
plt.axis('off')

# Passo (i) - Tons de cinza
plt.subplot(1, 4, 2)
plt.imshow(img_cinza, cmap='gray')
plt.title('(i) Tons de Cinza')
plt.axis('off')

# Passo (ii) - Desfocada
plt.subplot(1, 4, 3)
plt.imshow(img_desfocada, cmap='gray')
plt.title('(ii) Desfoque Gaussiano (21x21)')
plt.axis('off')

# Passo (iii) - Esboço final
plt.subplot(1, 4, 4)
plt.imshow(img_esboco, cmap='gray')
plt.title('(iii) Esboço a Lápis')
plt.axis('off')

plt.tight_layout()
plt.savefig('comparativo_esboco.png', dpi=150)
plt.show()

In [ ]:
# Visualizar resultados da correção gama
plt.figure(figsize=(20, 4))

# Imagem original
plt.subplot(2, 4, 1)
plt.imshow(imagem2, cmap='gray')
plt.title('Original')
plt.axis('off')

# Mostrar resultados para cada gamma
for i, gamma in enumerate(gammas):
    plt.subplot(2, 4, i+2)
    plt.imshow(resultados_gama[gamma], cmap='gray')
    plt.title(f'γ = {gamma}')
    plt.axis('off')

plt.tight_layout()
plt.savefig('comparativo_gama.png', dpi=150)
plt.show()

# Explicação dos efeitos:
print("\nEfeitos da correção gama:")
print("  γ < 1 : clareia a imagem (realça regiões escuras)")
print("  γ = 1 : sem alteração (imagem original)")
print("  γ > 1 : escurece a imagem (realça regiões claras)")

In [ ]:
# Carregar imagem2 usando OpenCV (apenas canal de intensidade)
imagem2 = cv2.imread('imagem2.jpg', cv2.IMREAD_GRAYSCALE)

if imagem2 is None:
    raise ValueError("Erro ao carregar imagem2.jpg!")

print(f"Imagem2 carregada: {imagem2.shape}")

# Valores de gamma para teste
gammas = [0.25, 0.5, 1.0, 1.5, 2.0, 3.0]

# Aplicar correção gama para cada valor
resultados_gama = {}
for gamma in gammas:
    resultados_gama[gamma] = correcao_gama(imagem2, gamma)
    cv2.imwrite(f'imagem2_gama_{gamma}.jpg', resultados_gama[gamma])
    print(f"Gamma={gamma}: imagem salva")

print("\nCorreção gama concluída!")

### Aplicar correção gama na imagem2 com diferentes valores de γ

In [ ]:
def correcao_gama(img, gamma):
    """
    Aplica correção gama manualmente.
    
    Parâmetros:
    - img: imagem monocromática de entrada [0, 255]
    - gamma: valor de gama (γ > 1 escurece, γ < 1 clareia)
    
    Retorna:
    - imagem com correção gama aplicada
    """
    # (i) Converter de [0, 255] para [0, 1]
    img_normalizada = img.astype(np.float64) / 255.0
    
    # (ii) Aplicar equação B = A^(1/γ)
    img_corrigida = np.power(img_normalizada, 1.0 / gamma)
    
    # (iii) Converter de volta para [0, 255]
    img_resultado = np.clip(img_corrigida * 255.0, 0, 255).astype(np.uint8)
    
    return img_resultado

---

## Questão 2: Correção Gama

Aplicar correção gama para ajustar o brilho de uma imagem monocromática.

Fórmula: $B = A^{(1/\gamma)}$

Etapas:
1. Converter intensidades de [0, 255] para [0, 1]
2. Aplicar $B = A^{(1/\gamma)}$
3. Converter resultado de volta para [0, 255]